# 05 · Extended pre-Haiyan baseline: ingestion and combined access

Process only the eight new baseline TIFFs: four VNP46A1 and four VNP46A2 files. This notebook reuses Notebook 00's band-label parser, cleaning, Zarr v2 schema, compression, processing flags and source-to-store QC.

**Outputs:** two baseline-only Zarr stores under `../datasets/VNP46/processed/baseline/`, plus lazily combined `ds_a1` and `ds_a2` for characterization. The combined datasets read the new and existing stores together by date; they are not an additional full Zarr copy. Existing stores and Notebook 00 outputs remain unchanged.

Place this notebook beside Notebook 00 in your project's notebook folder. Run top to bottom using the same environment. Keep all source TIFFs in `datasets/VNP46`. Dates are parsed from bands; filename endpoints do not establish observation availability.

Processing preserves all source bands. Reliability qualification (`Mandatory_Quality_Flag == 0`) is applied only when constructing the analysis view at the end. No missing observations are interpolated.


## 1. Imports and paths
The optional installation line is needed only if dependencies are missing. `dask[array]` enables combined access without loading the raster cubes into memory.


In [ ]:
# %pip install "zarr<3" numcodecs rasterio xarray "dask[array]" tqdm

from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import shutil
import time

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import xarray as xr
import zarr
from numcodecs import Blosc
from tqdm.auto import tqdm


In [ ]:
# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

DATA_DIR = Path("../datasets/VNP46")
OUTPUT_DIR = DATA_DIR / "processed" / "baseline"

A1_STORE = OUTPUT_DIR / "Haiyan_VNP46A1_baseline.zarr"
A2_STORE = OUTPUT_DIR / "Haiyan_VNP46A2_baseline.zarr"

MANIFEST_FILE = OUTPUT_DIR / "source_manifest.csv"
PILOT_QC_FILE = OUTPUT_DIR / "pilot_qc.csv"
PROCESSING_LOG_FILE = OUTPUT_DIR / "processing_log.csv"
PILOT_PASS_FILE = OUTPUT_DIR / "pilot_qc_passed.json"


# ------------------------------------------------------------
# Processing settings
# ------------------------------------------------------------

EVENT_DATE = pd.Timestamp("2013-11-08")

PILOT_DAYS = 5

CHUNK_Y = 512
CHUNK_X = 512

QC_WINDOW_SIZE = 128
QC_SAMPLE_STEP = 8

EXTRA_FILL_VALUES = (-9999.0, -999.0)

EXPECTED_PRODUCTS = ("VNP46A1", "VNP46A2")
EXPECTED_FILES_PER_PRODUCT = 4

# Missing source dates are reported; no observations are invented.
STRICT_DAILY_COVERAGE = False
REQUIRE_MATCHING_PRODUCT_DATES = True

# Applies only to the baseline stores created by this notebook.
RESET_STORES = False

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if int(zarr.__version__.split(".")[0]) != 2:
    raise RuntimeError(
        f"Detected zarr {zarr.__version__}. "
        'Install the stable v2 API with: %pip install "zarr<3"'
    )
EXISTING_STORE_PATHS = {
    "VNP46A1": DATA_DIR / "processed" / "Haiyan_VNP46A1.zarr",
    "VNP46A2": DATA_DIR / "processed" / "Haiyan_VNP46A2.zarr",
}

for path in EXISTING_STORE_PATHS.values():
    if not path.is_dir():
        raise FileNotFoundError(f"Existing Notebook 00 store not found: {path.resolve()}")


## 2. Discover and inspect the new baseline TIFFs
Only filenames containing `_Baseline_` are selected. The original pre-event and post-event TIFFs are excluded. The eight expected filenames are checked before reading band metadata.


In [ ]:
YMD_PATTERN = re.compile(
    r"(?<!\d)(20\d{2})[-_]?([01]\d)[-_]?([0-3]\d)(?!\d)"
)

DOY_PATTERN = re.compile(
    r"(?<!\d)A?(20\d{2})[-_]?([0-3]\d{2})(?!\d)"
)


def identify_product(path):
    match = re.search(r"VNP46A[12]", path.name.upper())

    if match is None:
        raise ValueError(f"Cannot identify product from filename: {path.name}")

    return match.group(0)


def parse_date_and_variable(label, band_index):
    label = str(label).strip()

    ymd_match = YMD_PATTERN.search(label)

    if ymd_match is not None:
        year, month, day = ymd_match.groups()
        date = pd.to_datetime(
            f"{year}-{month}-{day}",
            errors="coerce",
        )
        date_span = ymd_match.span()

    else:
        doy_match = DOY_PATTERN.search(label)

        if doy_match is None:
            return pd.NaT, None

        year, day_of_year = doy_match.groups()
        date = pd.to_datetime(
            f"{year}{day_of_year}",
            format="%Y%j",
            errors="coerce",
        )
        date_span = doy_match.span()

    if pd.isna(date):
        return pd.NaT, None

    variable = (
        label[:date_span[0]]
        + "_"
        + label[date_span[1]:]
    )

    variable = re.sub(
        r"(?<![A-Za-z0-9])VNP46A[12](?![A-Za-z0-9])",
        "_",
        variable,
        flags=re.IGNORECASE,
    )

    variable = YMD_PATTERN.sub("_", variable)
    variable = DOY_PATTERN.sub("_", variable)

    variable = re.sub(r"[^A-Za-z0-9_]+", "_", variable)
    variable = re.sub(r"_+", "_", variable).strip("_")

    if not variable:
        variable = f"band_{band_index:04d}"

    if variable[0].isdigit():
        variable = f"band_{variable}"

    if variable in {"date", "x", "y", "processed", "spatial_ref"}:
        variable = f"source_{variable}"

    return pd.Timestamp(date).normalize(), variable


source_files = sorted(
    set(DATA_DIR.glob("*_Baseline_*.tif"))
    | set(DATA_DIR.glob("*_Baseline_*.tiff"))
)

if not source_files:
    raise FileNotFoundError(
        f"No GeoTIFF files found in {DATA_DIR.resolve()}"
    )

print(f"Source directory: {DATA_DIR.resolve()}")
print(f"GeoTIFF files found: {len(source_files)}")

for path in source_files:
    print(f"  {path.name}: {path.stat().st_size / 1024**2:,.1f} MB")
baseline_periods = (
    "20120101_20120501",
    "20120501_20120901",
    "20120901_20130101",
    "20130101_20130512",
)
expected_baseline_stems = {
    f"Haiyan_{product}_{bands}_Baseline_{period}"
    for product, bands in (("VNP46A1", "KeyBands"), ("VNP46A2", "AllBands"))
    for period in baseline_periods
}
if {path.stem for path in source_files} != expected_baseline_stems:
    raise ValueError(
        "Baseline filenames differ from the eight expected downloads. "
        "Inspect source_files and baseline_periods before processing."
    )


In [ ]:
manifest_records = []
file_records = []

for path in source_files:
    product = identify_product(path)

    with rasterio.open(path) as src:
        transform_values = tuple(src.transform)[:6]

        file_records.append(
            {
                "product": product,
                "source_file": str(path),
                "source_name": path.name,
                "size_mb": path.stat().st_size / 1024**2,
                "band_count": src.count,
                "height": src.height,
                "width": src.width,
                "crs": src.crs.to_wkt() if src.crs else None,
                "transform": transform_values,
            }
        )

        for band_index in src.indexes:
            description = src.descriptions[band_index - 1]
            band_tags = src.tags(band_index)

            label = (
                description
                or band_tags.get("long_name")
                or band_tags.get("DESCRIPTION")
                or band_tags.get("description")
                or f"band_{band_index:04d}"
            )

            date, variable = parse_date_and_variable(
                label,
                band_index,
            )

            manifest_records.append(
                {
                    "product": product,
                    "source_file": str(path),
                    "source_name": path.name,
                    "band_index": band_index,
                    "description": label,
                    "date": date,
                    "variable": variable,
                    "dtype": src.dtypes[band_index - 1],
                    "nodata": src.nodatavals[band_index - 1],
                    "tags_json": json.dumps(
                        band_tags,
                        sort_keys=True,
                        default=str,
                    ),
                }
            )

manifest = pd.DataFrame(manifest_records)
file_inventory = pd.DataFrame(file_records)

manifest["date"] = pd.to_datetime(manifest["date"])

display(
    file_inventory[
        [
            "product",
            "source_name",
            "size_mb",
            "band_count",
            "height",
            "width",
        ]
    ]
)

display(manifest.head(20))


## 3. Validate dates, variables and grid
Retain Notebook 00's duplicate-band, grid, variable and A1/A2 matching checks. Calendar gaps are reported without blocking processing; absent dates are not filled. The Haiyan-date requirement is removed because these files contain baseline dates only. Inclusive overlapping export endpoints trigger a check rather than silent deduplication. Absent dates before the first actual band are not manufactured.


In [ ]:
# ------------------------------------------------------------
# 1. Expected products and file counts
# ------------------------------------------------------------

products_found = tuple(sorted(manifest["product"].unique()))

missing_products = sorted(
    set(EXPECTED_PRODUCTS) - set(products_found)
)

if missing_products:
    raise ValueError(
        f"Missing expected products: {missing_products}"
    )

file_counts = (
    file_inventory.groupby("product")["source_name"]
    .nunique()
    .sort_index()
)

print("Source files per product:")
display(file_counts.rename("files").to_frame())

unexpected_file_counts = file_counts[
    file_counts != EXPECTED_FILES_PER_PRODUCT
]

if not unexpected_file_counts.empty:
    raise ValueError(
        f"Expected {EXPECTED_FILES_PER_PRODUCT} baseline files per product, but found:\n"
        f"{unexpected_file_counts}"
    )


# ------------------------------------------------------------
# 2. Date parsing
# ------------------------------------------------------------

unparsed = manifest[manifest["date"].isna()].copy()

if not unparsed.empty:
    display(
        unparsed[
            [
                "product",
                "source_name",
                "band_index",
                "description",
            ]
        ].head(30)
    )

    raise ValueError(
        f"Could not parse dates for {len(unparsed):,} bands. "
        "Inspect the displayed raw band descriptions before processing."
    )


# ------------------------------------------------------------
# 3. Duplicate product-date-variable records
# ------------------------------------------------------------

duplicate_mask = manifest.duplicated(
    subset=["product", "date", "variable"],
    keep=False,
)

duplicates = manifest.loc[
    duplicate_mask,
    [
        "product",
        "date",
        "variable",
        "source_name",
        "band_index",
    ],
].sort_values(["product", "date", "variable"])

if not duplicates.empty:
    display(duplicates.head(50))

    raise ValueError(
        "Duplicate product-date-variable records detected. "
        "This may indicate overlapping export periods."
    )


# ------------------------------------------------------------
# 4. Consistent grid within each product
# ------------------------------------------------------------

product_grids = {}

for product, product_files in file_inventory.groupby("product"):
    reference = product_files.iloc[0]

    for _, row in product_files.iloc[1:].iterrows():
        same_shape = (
            row["height"] == reference["height"]
            and row["width"] == reference["width"]
        )

        same_crs = row["crs"] == reference["crs"]

        same_transform = np.allclose(
            np.asarray(row["transform"], dtype=float),
            np.asarray(reference["transform"], dtype=float),
            rtol=0,
            atol=1e-10,
        )

        if not (same_shape and same_crs and same_transform):
            raise ValueError(
                f"{product} grid mismatch:\n"
                f"Reference: {reference['source_name']}\n"
                f"Mismatch:  {row['source_name']}"
            )

    product_grids[product] = {
        "height": int(reference["height"]),
        "width": int(reference["width"]),
        "crs_wkt": reference["crs"],
        "transform": tuple(reference["transform"]),
    }


# ------------------------------------------------------------
# 5. Consistent variables across dates
# ------------------------------------------------------------

for product, product_manifest in manifest.groupby("product"):
    variable_sets = (
        product_manifest.groupby("date")["variable"]
        .apply(lambda values: tuple(sorted(values)))
    )

    reference_variables = variable_sets.iloc[0]

    inconsistent_dates = variable_sets[
        variable_sets != reference_variables
    ]

    if not inconsistent_dates.empty:
        display(inconsistent_dates.head(20))

        raise ValueError(
            f"{product} does not contain the same bands on every date."
        )


# ------------------------------------------------------------
# 6. Daily continuity
# ------------------------------------------------------------

for product, product_manifest in manifest.groupby("product"):
    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    expected_dates = pd.date_range(
        dates.min(),
        dates.max(),
        freq="D",
    )

    missing_dates = expected_dates.difference(dates)

    print(
        f"{product}: {len(dates):,} dates | "
        f"{dates.min().date()} to {dates.max().date()} | "
        f"{len(missing_dates):,} missing calendar dates"
    )

    if len(missing_dates):
        print(
            "First missing dates:",
            [date.strftime("%Y-%m-%d") for date in missing_dates[:20]],
        )

        if STRICT_DAILY_COVERAGE:
            raise ValueError(
                f"{product} has {len(missing_dates):,} missing dates."
            )


# ------------------------------------------------------------
# 7. Matching A1 and A2 date coverage
# ------------------------------------------------------------

date_sets = {
    product: set(
        pd.DatetimeIndex(
            product_manifest["date"].unique()
        )
    )
    for product, product_manifest
    in manifest.groupby("product")
}

if REQUIRE_MATCHING_PRODUCT_DATES:
    a1_only = sorted(
        date_sets["VNP46A1"] - date_sets["VNP46A2"]
    )

    a2_only = sorted(
        date_sets["VNP46A2"] - date_sets["VNP46A1"]
    )

    if a1_only or a2_only:
        print("Dates found only in VNP46A1:", a1_only[:20])
        print("Dates found only in VNP46A2:", a2_only[:20])

        raise ValueError(
            "VNP46A1 and VNP46A2 do not cover identical dates."
        )


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

source_summary = (
    manifest.groupby("product")
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        dates=("date", "nunique"),
        variables=("variable", "nunique"),
        source_bands=("band_index", "size"),
    )
)

display(source_summary)

for product, product_manifest in manifest.groupby("product"):
    variables = sorted(product_manifest["variable"].unique())

    print(f"\n{product} variables:")
    for variable in variables:
        print(f"  {variable}")

manifest.to_csv(MANIFEST_FILE, index=False)

print("\nSOURCE SANITY CHECKS: PASSED")
print(f"Manifest saved: {MANIFEST_FILE}")

# Check compatibility with existing stores before processing any baseline pixels.
for product in EXPECTED_PRODUCTS:
    existing_root = zarr.open_group(str(EXISTING_STORE_PATHS[product]), mode="r")
    grid = product_grids[product]
    if (
        existing_root.attrs["height"] != grid["height"]
        or existing_root.attrs["width"] != grid["width"]
        or existing_root.attrs["crs_wkt"] != grid["crs_wkt"]
    ):
        raise ValueError(f"{product}: baseline grid differs from the existing store.")
    np.testing.assert_allclose(
        existing_root.attrs["transform"], grid["transform"], rtol=0, atol=1e-10
    )
    baseline_variables = sorted(
        manifest.loc[manifest["product"].eq(product), "variable"].unique()
    )
    if sorted(existing_root.attrs["variables"]) != baseline_variables:
        raise ValueError(f"{product}: baseline bands differ from the existing store.")

print("BASELINE / EXISTING STORE COMPATIBILITY: PASSED")


## 4. Initialize resumable baseline stores
Same float32 arrays, `(date, y, x)` dimensions, one-day spatial chunks and Blosc Zstd level 5 / bitshuffle as Notebook 00. Existing baseline dates marked processed are skipped on reruns. Original Notebook 00 stores are never initialized or written here.


In [ ]:
COMPRESSOR = Blosc(
    cname="zstd",
    clevel=5,
    shuffle=Blosc.BITSHUFFLE,
)


def manifest_signature(product, product_manifest, grid):
    signature_table = product_manifest[
        [
            "source_name",
            "band_index",
            "date",
            "variable",
        ]
    ].copy()

    signature_table["date"] = (
        signature_table["date"].dt.strftime("%Y-%m-%d")
    )

    payload = {
        "product": product,
        "grid": {
            "height": grid["height"],
            "width": grid["width"],
            "crs_wkt": grid["crs_wkt"],
            "transform": list(grid["transform"]),
        },
        "manifest": signature_table.to_dict("records"),
    }

    encoded = json.dumps(
        payload,
        sort_keys=True,
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


def initialise_store(
    product,
    product_manifest,
    grid,
    store_path,
):
    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    variables = sorted(
        product_manifest["variable"].unique()
    )

    height = grid["height"]
    width = grid["width"]

    transform = grid["transform"]
    a, b, c, d, e, f = transform

    if not np.isclose(b, 0) or not np.isclose(d, 0):
        raise ValueError(
            f"{product} uses a rotated grid. "
            "One-dimensional x/y coordinates are not valid."
        )

    x = c + a * (np.arange(width) + 0.5)
    y = f + e * (np.arange(height) + 0.5)

    signature = manifest_signature(
        product,
        product_manifest,
        grid,
    )

    if RESET_STORES and store_path.exists():
        shutil.rmtree(store_path)

    if store_path.exists():
        root = zarr.open_group(
            str(store_path),
            mode="a",
        )

        existing_signature = root.attrs.get(
            "manifest_signature"
        )

        if existing_signature != signature:
            raise ValueError(
                f"{store_path.name} exists but does not match "
                "the current source manifest. Set RESET_STORES = True "
                "only if the existing processed store can be replaced."
            )

        print(f"Using existing store: {store_path}")
        return root

    root = zarr.open_group(
        str(store_path),
        mode="w",
    )

    root.attrs.update(
        {
            "product": product,
            "title": f"Haiyan {product} daily raster cube",
            "created_utc": datetime.utcnow().isoformat(),
            "manifest_signature": signature,
            "crs_wkt": grid["crs_wkt"],
            "transform": list(transform),
            "height": height,
            "width": width,
            "date_start": dates.min().strftime("%Y-%m-%d"),
            "date_end": dates.max().strftime("%Y-%m-%d"),
            "date_count": len(dates),
            "variables": variables,
            "chunk_shape": [1, CHUNK_Y, CHUNK_X],
            "missing_value": "NaN",
            "scaling_applied": False,
        }
    )

    date_array = root.create_dataset(
        "date",
        data=dates.values.astype("datetime64[ns]"),
        chunks=(min(len(dates), 1024),),
        compressor=None,
    )
    date_array.attrs["_ARRAY_DIMENSIONS"] = ["date"]

    x_array = root.create_dataset(
        "x",
        data=x.astype(np.float64),
        chunks=(min(width, 4096),),
        compressor=None,
    )
    x_array.attrs["_ARRAY_DIMENSIONS"] = ["x"]

    y_array = root.create_dataset(
        "y",
        data=y.astype(np.float64),
        chunks=(min(height, 4096),),
        compressor=None,
    )
    y_array.attrs["_ARRAY_DIMENSIONS"] = ["y"]

    processed_array = root.create_dataset(
        "processed",
        data=np.zeros(len(dates), dtype=np.uint8),
        chunks=(min(len(dates), 1024),),
        dtype="u1",
        fill_value=255,
        compressor=None,
    )
    processed_array.attrs.update(
        {
            "_ARRAY_DIMENSIONS": ["date"],
            "flag_values": [0, 1],
            "flag_meanings": "not_processed processed",
        }
    )

    spatial_ref = root.create_dataset(
        "spatial_ref",
        shape=(),
        dtype="i4",
        fill_value=0,
        compressor=None,
    )
    spatial_ref[()] = 0
    spatial_ref.attrs.update(
        {
            "_ARRAY_DIMENSIONS": [],
            "spatial_ref": grid["crs_wkt"],
            "crs_wkt": grid["crs_wkt"],
            "GeoTransform": f"{c} {a} {b} {f} {d} {e}",
        }
    )

    for variable in variables:
        variable_rows = product_manifest[
            product_manifest["variable"].eq(variable)
        ]

        source_descriptions = sorted(
            variable_rows["description"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        source_tags = (
            variable_rows["tags_json"]
            .dropna()
            .astype(str)
            .iloc[0]
        )

        array = root.create_dataset(
            variable,
            shape=(len(dates), height, width),
            chunks=(
                1,
                min(CHUNK_Y, height),
                min(CHUNK_X, width),
            ),
            dtype="f4",
            fill_value=np.nan,
            compressor=COMPRESSOR,
            dimension_separator="/",
        )

        array.attrs.update(
            {
                "_ARRAY_DIMENSIONS": ["date", "y", "x"],
                "grid_mapping": "spatial_ref",
                "source_descriptions": source_descriptions,
                "source_tags_example": source_tags,
                "scaling_applied": False,
            }
        )

    print(f"Created store: {store_path}")
    print(
        f"  Shape per variable: "
        f"({len(dates):,}, {height:,}, {width:,})"
    )
    print(f"  Variables: {len(variables)}")

    return root


STORE_PATHS = {
    "VNP46A1": A1_STORE,
    "VNP46A2": A2_STORE,
}

stores = {}

for product in EXPECTED_PRODUCTS:
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    stores[product] = initialise_store(
        product=product,
        product_manifest=product_manifest,
        grid=product_grids[product],
        store_path=STORE_PATHS[product],
    )


In [ ]:
def iter_windows(height, width, chunk_y, chunk_x):
    for row_off in range(0, height, chunk_y):
        window_height = min(
            chunk_y,
            height - row_off,
        )

        for col_off in range(0, width, chunk_x):
            window_width = min(
                chunk_x,
                width - col_off,
            )

            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=window_width,
                height=window_height,
            )


def clean_source_block(block):
    data = np.asarray(
        block.filled(np.nan),
        dtype=np.float32,
    )

    for fill_value in EXTRA_FILL_VALUES:
        data[data == fill_value] = np.nan

    data[~np.isfinite(data)] = np.nan

    return data


def process_dates(
    product,
    dates_to_process,
    skip_processed=True,
):
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    date_lookup = {
        pd.Timestamp(date): index
        for index, date in enumerate(dates)
    }

    root = zarr.open_group(
        str(STORE_PATHS[product]),
        mode="a",
    )

    height = int(root.attrs["height"])
    width = int(root.attrs["width"])

    records = []

    for date in tqdm(
        pd.DatetimeIndex(dates_to_process),
        desc=product,
    ):
        date = pd.Timestamp(date).normalize()
        date_index = date_lookup[date]

        if (
            skip_processed
            and root["processed"][date_index] == 1
        ):
            continue

        date_rows = product_manifest[
            product_manifest["date"].eq(date)
        ].sort_values("variable")

        if date_rows.empty:
            raise KeyError(
                f"No {product} source bands found for {date.date()}."
            )

        start_time = time.perf_counter()

        try:
            for source_file, file_rows in date_rows.groupby(
                "source_file",
                sort=False,
            ):
                file_rows = file_rows.sort_values("variable")

                band_indexes = (
                    file_rows["band_index"]
                    .astype(int)
                    .tolist()
                )

                variables = file_rows["variable"].tolist()

                with rasterio.open(source_file) as src:
                    for window in iter_windows(
                        height,
                        width,
                        CHUNK_Y,
                        CHUNK_X,
                    ):
                        block = src.read(
                            band_indexes,
                            window=window,
                            masked=True,
                            out_dtype="float32",
                        )

                        block = clean_source_block(block)

                        row_start = int(window.row_off)
                        row_stop = row_start + int(window.height)

                        col_start = int(window.col_off)
                        col_stop = col_start + int(window.width)

                        for band_position, variable in enumerate(
                            variables
                        ):
                            root[variable][
                                date_index,
                                row_start:row_stop,
                                col_start:col_stop,
                            ] = block[band_position]

            # Mark complete only after every band and window succeeds.
            root["processed"][date_index] = np.uint8(1)

        except Exception:
            root["processed"][date_index] = np.uint8(0)
            raise

        records.append(
            {
                "timestamp_utc": datetime.utcnow().isoformat(),
                "product": product,
                "date": date,
                "date_index": date_index,
                "elapsed_seconds": time.perf_counter() - start_time,
                "status": "processed",
            }
        )

    return pd.DataFrame(records)


def append_processing_log(records):
    if records.empty:
        return

    if PROCESSING_LOG_FILE.exists():
        existing = pd.read_csv(
            PROCESSING_LOG_FILE,
            parse_dates=["date"],
        )

        records = pd.concat(
            [existing, records],
            ignore_index=True,
        )

        records = records.drop_duplicates(
            subset=["product", "date"],
            keep="last",
        )

    records.to_csv(
        PROCESSING_LOG_FILE,
        index=False,
    )


## 5. Process and verify the first five available baseline dates
The QC compares source and stored pixels exactly and retains Notebook 00's flag and finite-value checks. If an early pilot has no finite values for a variable, inspect the source availability before proceeding; absence alone is not evidence of low nighttime lights.


In [ ]:
pilot_logs = []

pilot_dates_by_product = {}

for product in EXPECTED_PRODUCTS:
    product_dates = pd.DatetimeIndex(
        sorted(
            manifest.loc[
                manifest["product"].eq(product),
                "date",
            ].unique()
        )
    )

    pilot_dates = product_dates[:PILOT_DAYS]
    pilot_dates_by_product[product] = pilot_dates

    print(
        f"{product} pilot dates: "
        f"{pilot_dates.min().date()} to "
        f"{pilot_dates.max().date()}"
    )

    product_log = process_dates(
        product=product,
        dates_to_process=pilot_dates,
        skip_processed=True,
    )

    pilot_logs.append(product_log)

pilot_log = pd.concat(
    pilot_logs,
    ignore_index=True,
)

append_processing_log(pilot_log)

display(pilot_log)

print("FIVE-DAY PILOT PROCESSING: COMPLETE")


In [ ]:
def source_sample(row, window):
    with rasterio.open(row["source_file"]) as src:
        sample = src.read(
            int(row["band_index"]),
            window=window,
            masked=True,
            out_dtype="float32",
        )

    sample = clean_source_block(sample[np.newaxis, ...])[0]

    return sample


qc_records = []

for product in EXPECTED_PRODUCTS:
    product_manifest = manifest[
        manifest["product"].eq(product)
    ].copy()

    dates = pd.DatetimeIndex(
        sorted(product_manifest["date"].unique())
    )

    date_lookup = {
        pd.Timestamp(date): index
        for index, date in enumerate(dates)
    }

    root = zarr.open_group(
        str(STORE_PATHS[product]),
        mode="r",
    )

    height = int(root.attrs["height"])
    width = int(root.attrs["width"])

    sample_height = min(QC_WINDOW_SIZE, height)
    sample_width = min(QC_WINDOW_SIZE, width)

    row_off = max((height - sample_height) // 2, 0)
    col_off = max((width - sample_width) // 2, 0)

    audit_window = Window(
        col_off=col_off,
        row_off=row_off,
        width=sample_width,
        height=sample_height,
    )

    for date in pilot_dates_by_product[product]:
        date = pd.Timestamp(date).normalize()
        date_index = date_lookup[date]

        if root["processed"][date_index] != 1:
            raise RuntimeError(
                f"{product} {date.date()} was not marked processed."
            )

        date_rows = product_manifest[
            product_manifest["date"].eq(date)
        ].sort_values("variable")

        for _, row in date_rows.iterrows():
            variable = row["variable"]

            source_values = source_sample(
                row,
                audit_window,
            )

            stored_values = root[variable][
                date_index,
                row_off:row_off + sample_height,
                col_off:col_off + sample_width,
            ]

            np.testing.assert_allclose(
                stored_values,
                source_values,
                rtol=0,
                atol=0,
                equal_nan=True,
                err_msg=(
                    f"Source-to-store mismatch: "
                    f"{product}, {date.date()}, {variable}"
                ),
            )

            sampled = root[variable][
                date_index,
                ::QC_SAMPLE_STEP,
                ::QC_SAMPLE_STEP,
            ]

            finite = sampled[np.isfinite(sampled)]

            is_flag_band = any(
                token in variable.lower()
                for token in (
                    "flag",
                    "quality",
                    "qf_",
                    "cloud_mask",
                )
            )

            integer_like = True

            if is_flag_band and finite.size:
                integer_like = np.allclose(
                    finite,
                    np.round(finite),
                    rtol=0,
                    atol=1e-6,
                )

            qc_records.append(
                {
                    "product": product,
                    "date": date,
                    "variable": variable,
                    "sample_pixels": sampled.size,
                    "finite_pixels": finite.size,
                    "finite_fraction": (
                        finite.size / sampled.size
                        if sampled.size
                        else np.nan
                    ),
                    "minimum": (
                        float(np.min(finite))
                        if finite.size
                        else np.nan
                    ),
                    "median": (
                        float(np.median(finite))
                        if finite.size
                        else np.nan
                    ),
                    "maximum": (
                        float(np.max(finite))
                        if finite.size
                        else np.nan
                    ),
                    "flag_band": is_flag_band,
                    "integer_like": integer_like,
                    "source_match": True,
                }
            )

pilot_qc = pd.DataFrame(qc_records)

all_missing_variables = (
    pilot_qc.groupby(["product", "variable"])["finite_pixels"]
    .sum()
    .loc[lambda values: values == 0]
)

noninteger_flags = pilot_qc[
    pilot_qc["flag_band"]
    & ~pilot_qc["integer_like"]
]

if not all_missing_variables.empty:
    display(all_missing_variables)

    raise ValueError(
        "One or more variables contain no finite pixels "
        "across the five-day pilot."
    )

if not noninteger_flags.empty:
    display(noninteger_flags)

    raise ValueError(
        "A flag or quality band contains non-integer values."
    )

pilot_qc.to_csv(
    PILOT_QC_FILE,
    index=False,
)

pilot_pass = {
    "status": "passed",
    "timestamp_utc": datetime.utcnow().isoformat(),
    "pilot_days": PILOT_DAYS,
    "products": list(EXPECTED_PRODUCTS),
    "source_manifest": str(MANIFEST_FILE),
    "pilot_qc": str(PILOT_QC_FILE),
    "manifest_signatures": {
        product: stores[product].attrs["manifest_signature"]
        for product in EXPECTED_PRODUCTS
    },
}

PILOT_PASS_FILE.write_text(
    json.dumps(
        pilot_pass,
        indent=2,
    )
)

display(
    pilot_qc.groupby(["product", "variable"])
    .agg(
        pilot_finite_fraction=("finite_fraction", "mean"),
        pilot_minimum=("minimum", "min"),
        pilot_median=("median", "median"),
        pilot_maximum=("maximum", "max"),
        source_matches=("source_match", "all"),
    )
)

print("PILOT SANITY CHECKS: PASSED")
print(f"QC report: {PILOT_QC_FILE}")
print(f"Batch authorisation: {PILOT_PASS_FILE}")


## 6. Process the remaining baseline dates
Only the eight baseline TIFFs are in the processing manifest. Rerunning resumes unfinished dates. This step performs no copying or recompression of the existing event-period Zarr stores.


In [ ]:
RUN_FULL_BATCH = True


if RUN_FULL_BATCH:
    if not PILOT_PASS_FILE.exists():
        raise RuntimeError(
            "Full processing is blocked because the five-day "
            "pilot has not passed QC."
        )

    with PILOT_PASS_FILE.open() as file:
        pilot_authorisation = json.load(file)

    if pilot_authorisation.get("status") != "passed":
        raise RuntimeError(
            "The pilot QC authorisation is invalid."
        )

    if pilot_authorisation.get("manifest_signatures") != {
        product: stores[product].attrs["manifest_signature"]
        for product in EXPECTED_PRODUCTS
    }:
        raise RuntimeError("Rerun the pilot and QC for the current baseline manifest.")

    batch_logs = []

    for product in EXPECTED_PRODUCTS:
        product_dates = pd.DatetimeIndex(
            sorted(
                manifest.loc[
                    manifest["product"].eq(product),
                    "date",
                ].unique()
            )
        )

        product_log = process_dates(
            product=product,
            dates_to_process=product_dates,
            skip_processed=True,
        )

        batch_logs.append(product_log)

    batch_log = pd.concat(
        batch_logs,
        ignore_index=True,
    )

    append_processing_log(batch_log)

    final_status = []

    for product in EXPECTED_PRODUCTS:
        root = zarr.open_group(
            str(STORE_PATHS[product]),
            mode="r",
        )

        processed = root["processed"][:] == 1

        final_status.append(
            {
                "product": product,
                "processed_dates": int(processed.sum()),
                "total_dates": int(processed.size),
                "remaining_dates": int((~processed).sum()),
                "complete": bool(processed.all()),
            }
        )

    final_status = pd.DataFrame(final_status)
    display(final_status)

    if not final_status["complete"].all():
        raise RuntimeError(
            "The batch ended with unprocessed dates. "
            "Rerun this cell to resume."
        )

    for store_path in STORE_PATHS.values():
        zarr.consolidate_metadata(str(store_path))

    print("FULL BATCH PROCESSING: COMPLETE")

else:
    print(
        "Full batch not started. "
        "Set RUN_FULL_BATCH = True after reviewing the pilot QC."
    )


## 7. Combine baseline and existing stores by date
`ds_a1` and `ds_a2` are lazy, chronological analysis datasets. No merged raster copy is written. On future sessions, run imports, paths, and this section to reopen them without rerunning TIFF ingestion.

Shared dates use the existing store when its date is marked complete. A completed baseline date can replace an unfinished date left by an interrupted expansion. Unfinished existing dates without a baseline replacement stop the merge. The printed counts disclose shared dates and any replacements.

Zero radiance and zero-valued quality flags remain valid values: `mask_and_scale=False` preserves the representation written by Notebook 00.


In [ ]:
def open_combined_product(product, baseline_path, existing_path):
    baseline = xr.open_zarr(
        str(baseline_path), chunks={}, consolidated=False, mask_and_scale=False
    )
    existing = xr.open_zarr(
        str(existing_path), chunks={}, consolidated=False, mask_and_scale=False
    )

    for label, dataset in (("baseline", baseline), ("existing", existing)):
        dates = pd.DatetimeIndex(dataset["date"].values)
        if not dates.is_unique or not dates.is_monotonic_increasing:
            raise ValueError(f"{product} {label}: dates must be unique and sorted.")
        if dataset.attrs.get("product") != product:
            raise ValueError(f"Product metadata mismatch: {label}")
        if dataset["processed"].dims != ("date",):
            raise ValueError(f"Unexpected processing flags: {label}")

    if set(baseline.data_vars) != set(existing.data_vars):
        raise ValueError(f"{product}: baseline and existing variables differ.")
    for coordinate in ("x", "y"):
        np.testing.assert_array_equal(
            baseline[coordinate].values, existing[coordinate].values,
            err_msg=f"{product}: {coordinate} grid mismatch",
        )
    if baseline.attrs["crs_wkt"] != existing.attrs["crs_wkt"]:
        raise ValueError(f"{product}: CRS mismatch.")
    np.testing.assert_allclose(
        baseline.attrs["transform"], existing.attrs["transform"], rtol=0, atol=1e-10
    )
    for variable in baseline.data_vars:
        if (
            baseline[variable].dims != existing[variable].dims
            or baseline[variable].dtype != existing[variable].dtype
        ):
            raise ValueError(f"{product}: incompatible schema for {variable}.")

    baseline_dates = pd.DatetimeIndex(baseline["date"].values)
    existing_dates = pd.DatetimeIndex(existing["date"].values)
    if not np.all(baseline["processed"].values == 1):
        raise RuntimeError(f"{product}: finish baseline processing before combining.")

    existing_complete = existing["processed"].values == 1
    existing_kept_dates = existing_dates[existing_complete]
    unfinished = existing_dates[~existing_complete]
    if len(unfinished.difference(baseline_dates)):
        raise RuntimeError(f"{product}: existing store has unfinished dates outside the baseline.")

    added_dates = baseline_dates.difference(existing_kept_dates)
    shared_dates = baseline_dates.intersection(existing_kept_dates)
    combined = xr.concat(
        [baseline.sel(date=added_dates), existing.sel(date=existing_kept_dates)],
        dim="date", data_vars="minimal", coords="minimal",
        compat="equals", join="exact", combine_attrs="override",
    ).sortby("date")

    combined_dates = pd.DatetimeIndex(combined["date"].values)
    if not combined_dates.equals(baseline_dates.union(existing_dates).sort_values()):
        raise RuntimeError(f"{product}: combined date coverage is inconsistent.")

    combined.attrs = dict(existing.attrs)
    combined.attrs.pop("manifest_signature", None)
    combined.attrs.update({
        "title": f"Haiyan {product}: extended baseline and event-period data",
        "date_start": combined_dates.min().strftime("%Y-%m-%d"),
        "date_end": combined_dates.max().strftime("%Y-%m-%d"),
        "date_count": len(combined_dates),
        "source_stores": [str(baseline_path), str(existing_path)],
        "merge_policy": "Existing completed dates take precedence on overlap",
    })
    for variable in baseline.attrs["variables"]:
        combined[variable].attrs = dict(existing[variable].attrs)
        combined[variable].attrs["source_descriptions"] = sorted(set(
            baseline[variable].attrs.get("source_descriptions", [])
            + existing[variable].attrs.get("source_descriptions", [])
        ))

    print(
        f"{product}: {len(combined_dates):,} dates | "
        f"{combined_dates.min().date()} to {combined_dates.max().date()} | "
        f"{len(added_dates):,} supplied by baseline | "
        f"{len(shared_dates):,} shared dates retained from existing | "
        f"{len(unfinished):,} unfinished existing dates replaced"
    )
    return combined


ds_a1 = open_combined_product("VNP46A1", A1_STORE, EXISTING_STORE_PATHS["VNP46A1"])
ds_a2 = open_combined_product("VNP46A2", A2_STORE, EXISTING_STORE_PATHS["VNP46A2"])
np.testing.assert_array_equal(ds_a1["date"].values, ds_a2["date"].values)

display(ds_a1)
display(ds_a2)


## 8. Starting view for baseline characterization
The default characterization window is the newly downloaded interval, ending **2013-05-11**, immediately before Notebook 00's original start date. Change these explicit bounds to study a different baseline; downloading more data should not silently redefine it.

`dnb_baseline` retains DNB-BRDF as stored. `dnb_baseline_rq` keeps fresh/high-quality observations (`MQF == 0`) with finite, nonnegative radiance. It is a starting pixel-level reliability-qualified NTL view, not a settlement mask or daily valid-pixel coverage threshold. Retain missingness and assess coverage before interpreting baseline behaviour. Gap-filled NTL is available only as a diagnostic comparison.


In [ ]:
BASELINE_START = "2012-01-01"
BASELINE_END = "2013-05-11"

ds_a1_baseline = ds_a1.sel(date=slice(BASELINE_START, BASELINE_END))
ds_a2_baseline = ds_a2.sel(date=slice(BASELINE_START, BASELINE_END))

if ds_a2_baseline.sizes["date"] == 0:
    raise ValueError("No actual observations fall inside the selected baseline window.")

dnb_baseline = ds_a2_baseline["DNB_BRDF_Corrected_NTL"]
mqf_baseline = ds_a2_baseline["Mandatory_Quality_Flag"]
gapfilled_baseline = ds_a2_baseline["Gap_Filled_DNB_BRDF_Corrected_NTL"]

dnb_baseline_rq = dnb_baseline.where(
    (mqf_baseline == 0) & np.isfinite(dnb_baseline) & (dnb_baseline >= 0)
)

actual_dates = pd.DatetimeIndex(ds_a2_baseline["date"].values)
print(f"Requested baseline: {BASELINE_START} to {BASELINE_END}")
print(f"Available baseline: {actual_dates.min().date()} to {actual_dates.max().date()}")
print(f"Available dates: {len(actual_dates):,}")
print("Combined datasets and baseline views are ready; raster values remain lazily loaded.")
